# Milestone 5: Fine-tuning a pretrained vision model
*AI from scratch, milestone 5* · combine notebook for Stage 5

Transfer learning on real photos:
1. Take a modern pretrained vision backbone (`MobileNetV3-Small`, trained on 1.2M ImageNet photos).
2. Freeze its convolutional feature extractor and replace its final classification head for a new 4-class task.
3. Fine-tune the classifier head using Adam on real photos.
4. Measure per-class performance with precision, recall, and a full confusion matrix.

Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)`
are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['transform = T.Compose([T.Resize((112, 112)), T.ToTensor(), T.Normalize(mean, std)])'], 2: ['model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)', 'for p in model.parameters(): p.requires_grad = False', 'model.classifier[3] = nn.Linear(1024, 4)'], 3: ['for epoch in range(3): for i in range(0, len(X_tr), 64): bx, by = X_tr[i:i+64].to(device), y_tr[i:i+64].to(device); optimizer.zero_grad(); loss = criterion(model(bx), by); loss.backward(); optimizer.step()'], 4: ['model.eval(); with torch.no_grad(): test_preds = model(X_te.to(device)).argmax(dim=1).cpu().numpy()', 'cm = confusion_matrix(y_te.numpy(), test_preds); precision = precision_score(y_te.numpy(), test_preds, average=None); recall = recall_score(y_te.numpy(), test_preds, average=None)'], 5: ['f1_per_class = 2 * (precision * recall) / (precision + recall + 1e-12)', 'f1 = float(np.mean(f1_per_class))']}
_ANSWERS = {1: 'dHJhbnNmb3JtID0gVC5Db21wb3NlKFsKICAgIFQuUmVzaXplKCgxMTIsIDExMikpLAogICAgVC5Ub1RlbnNvcigpLAogICAgVC5Ob3JtYWxpemUobWVhbiwgc3RkKQpdKQ==', 2: 'bW9kZWwgPSBtb2JpbGVuZXRfdjNfc21hbGwod2VpZ2h0cz1Nb2JpbGVOZXRfVjNfU21hbGxfV2VpZ2h0cy5ERUZBVUxUKQpmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCk6CiAgICBwLnJlcXVpcmVzX2dyYWQgPSBGYWxzZQptb2RlbC5jbGFzc2lmaWVyWzNdID0gbm4uTGluZWFyKDEwMjQsIDQp', 3: 'bW9kZWwudHJhaW4oKQpmb3IgZXBvY2ggaW4gcmFuZ2UoMyk6CiAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4oWF90ciksIDY0KToKICAgICAgICBieCA9IFhfdHJbaTppICsgNjRdLnRvKGRldmljZSkKICAgICAgICBieSA9IHlfdHJbaTppICsgNjRdLnRvKGRldmljZSkKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICBsb3NzID0gY3JpdGVyaW9uKG1vZGVsKGJ4KSwgYnkpCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQ==', 4: 'bW9kZWwuZXZhbCgpCndpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgdGVzdF9wcmVkcyA9IG1vZGVsKFhfdGUudG8oZGV2aWNlKSkuYXJnbWF4KGRpbT0xKS5jcHUoKS5udW1weSgpCmNtID0gY29uZnVzaW9uX21hdHJpeCh5X3RlLm51bXB5KCksIHRlc3RfcHJlZHMpCnByZWNpc2lvbiA9IHByZWNpc2lvbl9zY29yZSh5X3RlLm51bXB5KCksIHRlc3RfcHJlZHMsIGF2ZXJhZ2U9Tm9uZSkKcmVjYWxsID0gcmVjYWxsX3Njb3JlKHlfdGUubnVtcHkoKSwgdGVzdF9wcmVkcywgYXZlcmFnZT1Ob25lKQ==', 5: 'ZjFfcGVyX2NsYXNzID0gMiAqIChwcmVjaXNpb24gKiByZWNhbGwpIC8gKHByZWNpc2lvbiArIHJlY2FsbCArIDFlLTEyKQpmMSA9IGZsb2F0KG5wLm1lYW4oZjFfcGVyX2NsYXNzKSk='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import io, urllib.request
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torchvision
from torchvision import transforms as T
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.metrics import confusion_matrix, precision_score, recall_score

torch.manual_seed(42)
np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load photo dataset (CIFAR-10 test parquet from Hugging Face hub)
url = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/test-00000-of-00001.parquet"
df = pd.read_parquet(io.BytesIO(urllib.request.urlopen(url).read()))

# Focus on 4 distinct categories: 0: plane, 1: car, 2: bird, 3: cat
sub = df[df["label"].isin([0, 1, 2, 3])].reset_index(drop=True)
imgs = [Image.open(io.BytesIO(r["bytes"])).convert("RGB") for r in sub["img"]]
labels = sub["label"].to_numpy()
class_names = ["plane", "car", "bird", "cat"]

# Split into 600 training photos and 200 testing photos
tr_idx, te_idx = np.arange(600), np.arange(600, 800)
print(f"Data ready: {len(tr_idx)} train photos, {len(te_idx)} test photos across {class_names}")

## 1. Image preprocessing transforms
Pretrained PyTorch models expect images normalized with ImageNet channel means and standard deviations:
Mean: `[0.485, 0.456, 0.406]`, Std: `[0.229, 0.224, 0.225]`.
Build `transform`: a `T.Compose` that:
- Resizes images to `(112, 112)`
- Converts to Tensor (`T.ToTensor()`)
- Normalizes with ImageNet mean and std (`T.Normalize(mean, std)`)
Assign to `transform`.

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# YOUR CODE HERE (stuck? hint(1), then answer(1))
...

In [ ]:
check(1, lambda: transform is not None and transform(imgs[0]).shape == (3, 112, 112), "transform resizes images to (3, 112, 112) and normalizes channels")

## 2. Freezing the backbone and adapting the head
Load `mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)`.
Freeze all feature backbone parameters by setting `p.requires_grad = False`.
Replace the final linear classification layer:
`model.classifier[3] = nn.Linear(1024, 4)`
Assign the model to `model`.

In [ ]:
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...

In [ ]:
check(2, lambda: sum(p.numel() for p in model.parameters() if p.requires_grad) == 1024 * 4 + 4 and model.classifier[3].out_features == 4, "backbone is frozen (0 gradients) and classifier head has 4,100 trainable parameters")

## 3. Fine-tuning the classifier
Train `model.classifier[3]` for 3 epochs with Adam (`lr=0.005`) and `CrossEntropyLoss`.
Pre-transform training and test images into batched tensors.
Store the fine-tuned model in `model`.

In [ ]:
if "model" in locals() and model is not None:
    X_tr = torch.stack([transform(imgs[i]) for i in tr_idx])
    y_tr = torch.tensor(labels[tr_idx], dtype=torch.int64)
    X_te = torch.stack([transform(imgs[i]) for i in te_idx])
    y_te = torch.tensor(labels[te_idx], dtype=torch.int64)

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.classifier[3].parameters(), lr=0.005)

    # YOUR CODE HERE (stuck? hint(3), then answer(3))
    ...

In [ ]:
check(3, lambda: (model(X_te.to(device)).argmax(dim=1).cpu().numpy() == y_te.numpy()).mean() > 0.70, "Transfer learning achieves > 70% accuracy on fine-tuning with only 600 photos")

## 4. Per-class precision and recall
Overall accuracy hides per-category weaknesses (e.g. confusing cats with birds).
Compute:
- `cm`: 4x4 confusion matrix using `confusion_matrix(y_te.numpy(), test_preds)`
- `precision`: 4-element array of precision scores per class: `precision_score(y_te.numpy(), test_preds, average=None)`
- `recall`: 4-element array of recall scores per class: `recall_score(y_te.numpy(), test_preds, average=None)`

In [ ]:
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...

if "precision" in locals() and precision is not None:
    for name, p, r in zip(class_names, precision, recall):
        print(f"Class '{name}': Precision = {p:.2f}, Recall = {r:.2f}")

In [ ]:
check(4, lambda: cm.shape == (4, 4) and len(precision) == 4 and len(recall) == 4 and (precision > 0.5).all(), "confusion matrix and per-class precision and recall calculated across all 4 categories")

## 5. Macro F1-score
Write `macro_f1`: the arithmetic mean of each class's harmonic mean ($F_1 = 2 \frac{P \cdot R}{P + R}$).
Compute and store in `f1`.

In [ ]:
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...

if "f1" in locals() and f1 is not None:
    print(f"Macro F1-Score: {f1:.3f}")

In [ ]:
check(5, lambda: f1 > 0.65, "Macro F1 balances precision and recall across all categories equally")